In [34]:
# ============================================================
# CELL 24
# Save Unlabeled Text Embeddings
# ============================================================


print("=" * 90)
print("SAVE UNLABELED TEXT EMBEDDINGS")
print("=" * 90)


EMBED_SAVE_DIR = (
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr"
)


EMBED_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


UNLABELED_TEXT_EMBED_PATH = (
    EMBED_SAVE_DIR /
    "unlabeled_description_embeddings.npy"
)


np.save(
    UNLABELED_TEXT_EMBED_PATH,
    unlabeled_text_embeddings
)


print(
    "Saved:",
    UNLABELED_TEXT_EMBED_PATH
)


print(
    "Shape:",
    unlabeled_text_embeddings.shape
)

SAVE UNLABELED TEXT EMBEDDINGS
Saved: ..\final_data\embeddings\xlmr\unlabeled_description_embeddings.npy
Shape: (18331, 768)


In [2]:
print("="*90)
print("19K MODALITY AVAILABILITY")
print("="*90)


availability_cols = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]


for col in availability_cols:
    
    print(
        f"{col}:",
        true_unlabeled[col].sum(),
        "/",
        len(true_unlabeled),
        f"({true_unlabeled[col].mean()*100:.2f}%)"
    )

19K MODALITY AVAILABILITY
has_profile: 18331 / 18331 (100.00%)
has_description: 11647 / 18331 (63.54%)
has_raw_tweets: 0 / 18331 (0.00%)
has_behavior_temporal: 18331 / 18331 (100.00%)
has_graph: 3033 / 18331 (16.55%)


In [3]:
import pickle
import numpy as np


print("="*90)
print("GRAPH COVERAGE CHECK")
print("="*90)


GRAPH_DIR = BASE_DIR / "final_data" / "graph"


# load node mapping

with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:
    node_to_id = pickle.load(f)


print(
    "Graph nodes:",
    len(node_to_id)
)


# match with 19k users

user_keys = (
    true_unlabeled["user_key"]
    .astype(str)
    .str.lower()
)


graph_keys = set(
    str(k).lower()
    for k in node_to_id.keys()
)


matched = user_keys.isin(graph_keys)


print(
    "Matched users:",
    matched.sum()
)


print(
    "Unmatched users:",
    (~matched).sum()
)


print(
    "Coverage:",
    matched.mean()*100
)

GRAPH COVERAGE CHECK
Graph nodes: 13466
Matched users: 12397
Unmatched users: 5934
Coverage: 67.62860727728983


In [4]:
import numpy as np
import pickle


print("="*90)
print("GRAPH FEATURE ALIGNMENT CHECK")
print("="*90)


GRAPH_DIR = BASE_DIR / "final_data" / "graph"


# load node mapping

with open(
    GRAPH_DIR / "node_to_id.pkl",
    "rb"
) as f:
    node_to_id = pickle.load(f)


degree_features = np.load(
    GRAPH_DIR / "degree_features.npy"
)


print(
    "Nodes in mapping:",
    len(node_to_id)
)


print(
    "Degree feature shape:",
    degree_features.shape
)


print(
    "Feature dimension:",
    degree_features.shape[1]
)


# Check matched node ids

matched_nodes = []


for key in (
    true_unlabeled["user_key"]
    .astype(str)
    .str.lower()
):

    if key in node_to_id:
        matched_nodes.append(
            node_to_id[key]
        )


matched_nodes = np.array(
    matched_nodes
)


print(
    "\nMatched node ids:",
    len(matched_nodes)
)


print(
    "Max node id:",
    matched_nodes.max()
)


print(
    "Valid node ids:",
    np.all(
        matched_nodes < len(degree_features)
    )
)

GRAPH FEATURE ALIGNMENT CHECK
Nodes in mapping: 13466
Degree feature shape: (13466, 3)
Feature dimension: 3

Matched node ids: 5560
Max node id: 13465
Valid node ids: True


In [5]:
print("="*90)
print("BUILD GRAPH ALIGNMENT MAP")
print("="*90)


# normalize graph keys

normalized_node_to_id = {
    str(k).lower(): v
    for k, v in node_to_id.items()
}


graph_node_ids = []


matched_count = 0


for user_key in (
    true_unlabeled["user_key"]
    .astype(str)
    .str.lower()
):

    if user_key in normalized_node_to_id:

        graph_node_ids.append(
            normalized_node_to_id[user_key]
        )

        matched_count += 1

    else:

        graph_node_ids.append(
            -1
        )



true_unlabeled["graph_node_id"] = graph_node_ids



print(
    "Matched users:",
    matched_count
)


print(
    "Coverage:",
    matched_count / len(true_unlabeled) * 100
)


print(
    "\nSample:"
)


print(
    true_unlabeled[
        [
            "user_key",
            "graph_node_id"
        ]
    ].head(10)
)

BUILD GRAPH ALIGNMENT MAP
Matched users: 12397
Coverage: 67.62860727728983

Sample:
          user_key  graph_node_id
0  patriotpointman             -1
1     iranziba1401             -1
2      miiran20194           4005
3    artemis540721           2463
4       tifraghe_n          12771
5   estoucomisrael          10128
6         jjauthor           7537
7      lillian3003           8534
8   amiralikohrang           6027
9    limportant_fr          12263


In [6]:
import numpy as np


print("="*90)
print("BUILD UNLABELED GRAPH FEATURES")
print("="*90)


graph_features_unlabeled = np.zeros(
    (
        len(true_unlabeled),
        degree_features.shape[1]
    ),
    dtype=np.float32
)


for idx, node_id in enumerate(
    true_unlabeled["graph_node_id"]
):

    if node_id != -1:

        graph_features_unlabeled[idx] = (
            degree_features[node_id]
        )


print(
    "Graph feature matrix shape:"
)

print(
    graph_features_unlabeled.shape
)


print(
    "\nUsers with graph features:"
)

print(
    np.sum(
        true_unlabeled["graph_node_id"] != -1
    )
)


print(
    "\nUsers without graph features:"
)

print(
    np.sum(
        true_unlabeled["graph_node_id"] == -1
    )
)


print(
    "\nSample features:"
)

print(
    graph_features_unlabeled[:5]
)

BUILD UNLABELED GRAPH FEATURES
Graph feature matrix shape:
(18331, 3)

Users with graph features:
12397

Users without graph features:
5934

Sample features:
[[ 0.  0.  0.]
 [ 0.  0.  0.]
 [33. 33. 66.]
 [31. 31. 62.]
 [ 1.  1.  2.]]


In [9]:
import joblib


print("="*90)
print("LOAD FROZEN TABULAR PREPROCESSING")
print("="*90)


PREPROCESS_DIR = (
    BASE_DIR /
    "models" /
    "preprocessing"
)


IMPUTER_PATH = (
    PREPROCESS_DIR /
    "numeric_median_imputer.joblib"
)


SCALER_PATH = (
    PREPROCESS_DIR /
    "robust_scaler.joblib"
)


numeric_imputer = joblib.load(
    IMPUTER_PATH
)


robust_scaler = joblib.load(
    SCALER_PATH
)


print(
    "Imputer loaded:",
    type(numeric_imputer)
)


print(
    "Scaler loaded:",
    type(robust_scaler)
)

LOAD FROZEN TABULAR PREPROCESSING
Imputer loaded: <class 'sklearn.impute._base.SimpleImputer'>
Scaler loaded: <class 'sklearn.preprocessing._data.RobustScaler'>


In [12]:
import numpy as np


print("="*90)
print("PREPARE TABULAR FEATURES - CORRECT PIPELINE")
print("="*90)


# features expected by imputer

IMPUTER_FEATURES = (
    numeric_imputer.feature_names_in_
    .tolist()
)


print(
    "Imputer input features:",
    len(IMPUTER_FEATURES)
)


# select

unlabeled_numeric = (
    true_unlabeled[
        IMPUTER_FEATURES
    ]
    .copy()
)


print(
    "Before imputation:",
    unlabeled_numeric.shape
)


# impute

unlabeled_numeric = (
    numeric_imputer
    .transform(unlabeled_numeric)
)


print(
    "After imputation:",
    unlabeled_numeric.shape
)

PREPARE TABULAR FEATURES - CORRECT PIPELINE
Imputer input features: 36
Before imputation: (18331, 36)
After imputation: (18331, 36)


In [13]:
print("="*90)
print("RECONSTRUCT FINAL 40 TABULAR FEATURES")
print("="*90)


# convert imputed array back to dataframe

unlabeled_numeric_df = pd.DataFrame(
    unlabeled_numeric,
    columns=IMPUTER_FEATURES
)


# log transformation

unlabeled_numeric_df = np.log1p(
    unlabeled_numeric_df
)


# remove redundant features

DROP_FEATURES = [
    "status_count",
    "normal_followers_count"
]


unlabeled_numeric_df = (
    unlabeled_numeric_df
    .drop(
        columns=DROP_FEATURES
    )
)


print(
    "After removing redundant features:"
)

print(
    unlabeled_numeric_df.shape
)



# add binary features

BINARY_FEATURES = [
    "default_profile",
    "default_profile_image",
    "has_custom_timelines",
    "possibly_sensitive",
    "hashtag_in_description",
    "numbers_in_description"
]


for col in BINARY_FEATURES:
    
    unlabeled_numeric_df[col] = (
        true_unlabeled[col]
        .values
    )


print(
    "\nBefore scaling:"
)

print(
    unlabeled_numeric_df.shape
)

RECONSTRUCT FINAL 40 TABULAR FEATURES
After removing redundant features:
(18331, 34)

Before scaling:
(18331, 40)


c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


In [14]:
print("="*90)
print("CHECK INVALID VALUES AFTER LOG1P")
print("="*90)


invalid_report = pd.DataFrame({
    "nan_count": unlabeled_numeric_df.isna().sum(),
    "inf_count": np.isinf(
        unlabeled_numeric_df
    ).sum()
})


invalid_report = (
    invalid_report[
        (invalid_report["nan_count"] > 0)
        |
        (invalid_report["inf_count"] > 0)
    ]
)


invalid_report

CHECK INVALID VALUES AFTER LOG1P


,nan_count,inf_count
retweet_as_tweet_rate,27,3


In [15]:
print("="*90)
print("FIX INVALID VALUES BEFORE LOG1P")
print("="*90)


# recreate dataframe after imputation

unlabeled_numeric_df = pd.DataFrame(
    unlabeled_numeric,
    columns=IMPUTER_FEATURES
)


# replace inf values

unlabeled_numeric_df = (
    unlabeled_numeric_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


print(
    "NaN after replacing inf:"
)

print(
    unlabeled_numeric_df.isna()
    .sum()
    .sum()
)


# fill again using median values from fitted imputer

unlabeled_numeric_df = pd.DataFrame(
    numeric_imputer.transform(
        unlabeled_numeric_df
    ),
    columns=IMPUTER_FEATURES
)


print(
    "After second imputation:"
)

print(
    unlabeled_numeric_df.isna()
    .sum()
    .sum()
)

FIX INVALID VALUES BEFORE LOG1P
NaN after replacing inf:
0
After second imputation:
0


In [17]:
print("="*90)
print("CHECK SCALER FEATURES")
print("="*90)


print(
    "Number of scaler features:"
)

print(
    len(robust_scaler.feature_names_in_)
)


print(
    "\nScaler features:"
)

print(
    robust_scaler.feature_names_in_.tolist()
)

CHECK SCALER FEATURES
Number of scaler features:
33

Scaler features:
['followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_rate_per_tweet', 'mean_user_mentions_per_tweet', 'retweet_as_tweet_rate', 'no_retweet_tweets', 'mean_retweets_per_tweet', 'description_length', 'followers_friend_ratio', 'num_digits_in_name', 'num_digits_in_username']


In [18]:
print("="*90)
print("FIND UNSCALED FEATURES")
print("="*90)


MODEL_FEATURES = TABULAR_FEATURES


scaler_features = (
    robust_scaler
    .feature_names_in_
    .tolist()
)


remaining_features = [
    f for f in MODEL_FEATURES
    if f not in scaler_features
]


print(
    "Features not scaled:"
)

print(
    remaining_features
)


print(
    "\nCount:"
)

print(
    len(remaining_features)
)

FIND UNSCALED FEATURES
Features not scaled:
['default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'url_in_description']

Count:
7


In [20]:
print("="*90)
print("REBUILD CLEAN NUMERIC PIPELINE")
print("="*90)


# start from imputed data again

numeric_df_clean = pd.DataFrame(
    unlabeled_numeric,
    columns=IMPUTER_FEATURES
)


# replace invalid values before log

numeric_df_clean = (
    numeric_df_clean
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# second safety imputation

numeric_df_clean = pd.DataFrame(
    numeric_imputer.transform(
        numeric_df_clean
    ),
    columns=IMPUTER_FEATURES
)


# log1p

numeric_df_clean = np.log1p(
    numeric_df_clean
)


# check

print(
    "NaN after log:",
    np.isnan(
        numeric_df_clean.values
    ).sum()
)


print(
    "Inf after log:",
    np.isinf(
        numeric_df_clean.values
    ).sum()
)

REBUILD CLEAN NUMERIC PIPELINE
NaN after log: 27
Inf after log: 3


c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
c:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\env\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


In [21]:
print("="*90)
print("FIX VALUES BEFORE LOG1P")
print("="*90)


# start again from imputed data

numeric_df_clean = pd.DataFrame(
    unlabeled_numeric,
    columns=IMPUTER_FEATURES
)


# replace inf

numeric_df_clean = (
    numeric_df_clean
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# fill missing

numeric_df_clean = pd.DataFrame(
    numeric_imputer.transform(
        numeric_df_clean
    ),
    columns=IMPUTER_FEATURES
)


# check minimum values before log

print(
    "Minimum values before log:"
)

print(
    numeric_df_clean.min()
    .sort_values()
    .head(10)
)

FIX VALUES BEFORE LOG1P
Minimum values before log:
retweet_as_tweet_rate     -698.0
followers_count              0.0
favourites_count             0.0
friends_count                0.0
media_count                  0.0
normal_followers_count       0.0
friends_growth_rate          0.0
listed_count                 0.0
mean_no_media_per_tweet      0.0
no_type_reply                0.0
dtype: float64


In [22]:
print("="*90)
print("FIX RATE FEATURES")
print("="*90)


RATE_FEATURES = [
    "retweet_as_tweet_rate"
]


for col in RATE_FEATURES:
    
    numeric_df_clean[col] = (
        numeric_df_clean[col]
        .clip(lower=0)
    )


print(
    "Minimum after clipping:"
)

print(
    numeric_df_clean[RATE_FEATURES]
    .min()
)

FIX RATE FEATURES
Minimum after clipping:
retweet_as_tweet_rate    0.0
dtype: float64


In [23]:
print("="*90)
print("APPLY LOG1P AFTER FIX")
print("="*90)


numeric_df_clean = np.log1p(
    numeric_df_clean
)


print(
    "NaN after log:"
)

print(
    np.isnan(
        numeric_df_clean.values
    ).sum()
)


print(
    "Inf after log:"
)

print(
    np.isinf(
        numeric_df_clean.values
    ).sum()
)

APPLY LOG1P AFTER FIX
NaN after log:
0
Inf after log:
0


In [25]:
print("="*90)
print("FINAL TABULAR MATRIX CREATION")
print("="*90)


# convert back to dataframe after log

numeric_df_clean = pd.DataFrame(
    numeric_df_clean,
    columns=IMPUTER_FEATURES
)


# remove features not used in final model

DROP_FEATURES = [
    "status_count",
    "normal_followers_count"
]


numeric_df_clean = (
    numeric_df_clean
    .drop(
        columns=DROP_FEATURES
    )
)


print(
    "Numeric features after drop:"
)

print(
    numeric_df_clean.shape
)



# scale only 33 features

scaled_numeric = (
    robust_scaler.transform(
        numeric_df_clean[
            robust_scaler.feature_names_in_
        ]
    )
)


scaled_numeric_df = pd.DataFrame(
    scaled_numeric,
    columns=robust_scaler.feature_names_in_
)



# add 7 unscaled features

UNSCALED_FEATURES = [
    "default_profile",
    "default_profile_image",
    "has_custom_timelines",
    "possibly_sensitive",
    "hashtag_in_description",
    "numbers_in_description",
    "url_in_description"
]


unscaled_df = (
    true_unlabeled[
        UNSCALED_FEATURES
    ]
    .reset_index(drop=True)
)



# final concat

final_tabular_df = pd.concat(
    [
        scaled_numeric_df,
        unscaled_df
    ],
    axis=1
)


print(
    "\nFinal shape:"
)

print(
    final_tabular_df.shape
)


print(
    "\nNaN:"
)

print(
    final_tabular_df.isna().sum().sum()
)


print(
    "\nInf:"
)

print(
    np.isinf(
        final_tabular_df.values
    ).sum()
)

FINAL TABULAR MATRIX CREATION
Numeric features after drop:
(18331, 34)

Final shape:
(18331, 40)

NaN:
0

Inf:
0


In [27]:
print("="*90)
print("CHECK USER EMBEDDINGS")
print("="*90)


import numpy as np


train_user_emb = np.load(
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr" /
    "train_user_embeddings.npy"
)


train_user_keys = np.load(
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr" /
    "train_user_keys.npy",
    allow_pickle=True
)


print(
    "Embedding shape:"
)

print(
    train_user_emb.shape
)


print(
    "Keys shape:"
)

print(
    train_user_keys.shape
)


print(
    "Sample keys:"
)

print(
    train_user_keys[:5]
)

CHECK USER EMBEDDINGS
Embedding shape:
(671, 768)
Keys shape:
(671, 1)
Sample keys:
[['007mi00']
 ['0mhp037']
 ['2002kingfox']
 ['2expvzwq7llpx4p']
 ['2tosgzcfj8vtsjk']]


In [28]:
print("="*90)
print("PREPARE UNLABELED TEXT DATA")
print("="*90)


TEXT_COLUMN = "clean_description"


print(
    "Users with text:"
)

print(
    true_unlabeled[TEXT_COLUMN]
    .notna()
    .sum()
)


print(
    "\nEmpty texts:"
)

print(
    (
        true_unlabeled[TEXT_COLUMN]
        .fillna("")
        .str.strip()
        .eq("")
    )
    .sum()
)

PREPARE UNLABELED TEXT DATA
Users with text:
11441

Empty texts:
6890


In [29]:
print("="*90)
print("SEARCH XLM-R CONFIG")
print("="*90)


# اگر در notebook قبلی متغیرها load شده باشند

try:
    print("TEXT_MODEL_NAME:")
    print(TEXT_MODEL_NAME)
except:
    print("TEXT_MODEL_NAME not found")


try:
    print("MAX_LENGTH:")
    print(MAX_LENGTH)
except:
    print("MAX_LENGTH not found")

SEARCH XLM-R CONFIG
TEXT_MODEL_NAME:
TEXT_MODEL_NAME not found
MAX_LENGTH:
MAX_LENGTH not found


In [30]:
print("="*90)
print("INITIALIZE XLM-R TEXT ENCODER")
print("="*90)


from transformers import (
    AutoTokenizer,
    AutoModel
)

import torch


DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


TEXT_MODEL_NAME = (
    "xlm-roberta-base"
)


MAX_LENGTH = 128


tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)


text_encoder = AutoModel.from_pretrained(
    TEXT_MODEL_NAME
)


text_encoder = text_encoder.to(
    DEVICE
)


text_encoder.eval()


print(
    "Model:",
    TEXT_MODEL_NAME
)

print(
    "Device:",
    DEVICE
)

INITIALIZE XLM-R TEXT ENCODER


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: xlm-roberta-base
Device: cpu


In [31]:
# ============================================================
# CELL 22
# Define XLM-R Embedding Function
# ============================================================


from tqdm.auto import tqdm
import numpy as np


def generate_text_embeddings(
    texts,
    batch_size=32
):

    all_embeddings = []


    for start_idx in tqdm(
        range(
            0,
            len(texts),
            batch_size
        )
    ):

        batch_texts = texts[
            start_idx:
            start_idx + batch_size
        ]


        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )


        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }


        with torch.no_grad():

            outputs = text_encoder(
                **encoded
            )


        last_hidden_state = (
            outputs.last_hidden_state
        )


        attention_mask = (
            encoded["attention_mask"]
            .unsqueeze(-1)
        )


        masked_embeddings = (
            last_hidden_state *
            attention_mask
        )


        summed_embeddings = (
            masked_embeddings.sum(
                dim=1
            )
        )


        token_counts = (
            attention_mask.sum(
                dim=1
            )
        )


        mean_embeddings = (
            summed_embeddings /
            token_counts.clamp(
                min=1e-9
            )
        )


        all_embeddings.append(
            mean_embeddings
            .cpu()
            .numpy()
        )


    return np.vstack(
        all_embeddings
    )

In [32]:
# ============================================================
# CELL 23
# Generate Unlabeled Description Embeddings
# ============================================================


print("=" * 90)
print("GENERATING UNLABELED TEXT EMBEDDINGS")
print("=" * 90)


TEXT_COLUMN = "clean_description"


unlabeled_texts = (
    true_unlabeled[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)


print(
    "Number of texts:",
    len(unlabeled_texts)
)


unlabeled_text_embeddings = (
    generate_text_embeddings(
        unlabeled_texts,
        batch_size=32
    )
)


print(
    "\nEmbedding shape:",
    unlabeled_text_embeddings.shape
)

GENERATING UNLABELED TEXT EMBEDDINGS
Number of texts: 18331


  0%|          | 0/573 [00:00<?, ?it/s]


Embedding shape: (18331, 768)


In [35]:
# ============================================================
# CELL 24
# Save Unlabeled Text Embeddings
# ============================================================


print("=" * 90)
print("SAVE UNLABELED TEXT EMBEDDINGS")
print("=" * 90)


EMBED_SAVE_DIR = (
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr"
)


EMBED_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


UNLABELED_TEXT_EMBED_PATH = (
    EMBED_SAVE_DIR /
    "unlabeled_description_embeddings.npy"
)


np.save(
    UNLABELED_TEXT_EMBED_PATH,
    unlabeled_text_embeddings
)


print(
    "Saved:",
    UNLABELED_TEXT_EMBED_PATH
)


print(
    "Shape:",
    unlabeled_text_embeddings.shape
)

SAVE UNLABELED TEXT EMBEDDINGS
Saved: ..\final_data\embeddings\xlmr\unlabeled_description_embeddings.npy
Shape: (18331, 768)


In [36]:
# ============================================================
# CELL 25
# Apply Text Modality Mask
# ============================================================


print("=" * 90)
print("APPLY TEXT MODALITY MASK")
print("=" * 90)


text_available_mask = (
    true_unlabeled["clean_description"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)


print(
    "Users with text:",
    text_available_mask.sum()
)


print(
    "Users without text:",
    (~text_available_mask).sum()
)



# zero embedding for missing text modality

unlabeled_text_embeddings[
    ~text_available_mask.values
] = 0



print(
    "\nFinal text embedding shape:",
    unlabeled_text_embeddings.shape
)

APPLY TEXT MODALITY MASK
Users with text: 11441
Users without text: 6890

Final text embedding shape: (18331, 768)


In [40]:
# ============================================================
# CELL 26
# Build Unlabeled Multimodal Dataset
# ============================================================


print("=" * 90)
print("BUILD SEMI-SUPERVISED DATASET")
print("=" * 90)


import torch
from torch.utils.data import Dataset



class SemiSupervisedDataset(Dataset):

    def __init__(
        self,
        tabular,
        text,
        graph,
        user_keys
    ):

        self.tabular = torch.tensor(
            tabular,
            dtype=torch.float32
        )

        self.text = torch.tensor(
            text,
            dtype=torch.float32
        )

        self.graph = torch.tensor(
            graph,
            dtype=torch.float32
        )

        self.user_keys = user_keys


    def __len__(self):

        return len(self.user_keys)


    def __getitem__(
        self,
        idx
    ):

        return {
            "user_key":
                self.user_keys[idx],

            "tabular":
                self.tabular[idx],

            "text":
                self.text[idx],

            "graph":
                self.graph[idx]
        }



unlabeled_dataset = SemiSupervisedDataset(

    tabular=
        final_tabular_df.values,

    text=
        unlabeled_text_embeddings,

    graph=
        graph_features_unlabeled,

    user_keys=
        true_unlabeled["user_key"].values

)



print(
    "Dataset size:",
    len(unlabeled_dataset)
)



sample = unlabeled_dataset[0]


print("\nSample shapes:")

print(
    "Tabular:",
    sample["tabular"].shape
)

print(
    "Text:",
    sample["text"].shape
)

print(
    "Graph:",
    sample["graph"].shape
)

BUILD SEMI-SUPERVISED DATASET
Dataset size: 18331

Sample shapes:
Tabular: torch.Size([40])
Text: torch.Size([768])
Graph: torch.Size([3])


In [41]:
# ============================================================
# CELL 27
# Create Inference DataLoader
# ============================================================


from torch.utils.data import DataLoader


print("=" * 90)
print("CREATE UNLABELED INFERENCE DATALOADER")
print("=" * 90)



UNLABELED_BATCH_SIZE = 64


unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=UNLABELED_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)



print(
    "Number of batches:",
    len(unlabeled_loader)
)



batch = next(iter(unlabeled_loader))


print("\nBatch shapes:")

print(
    "Tabular:",
    batch["tabular"].shape
)

print(
    "Text:",
    batch["text"].shape
)

print(
    "Graph:",
    batch["graph"].shape
)

print(
    "User keys:",
    len(batch["user_key"])
)

CREATE UNLABELED INFERENCE DATALOADER
Number of batches: 287

Batch shapes:
Tabular: torch.Size([64, 40])
Text: torch.Size([64, 768])
Graph: torch.Size([64, 3])
User keys: 64


In [44]:
class FeatureEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(
                input_dim,
                output_dim
            ),
            nn.ReLU(),
            nn.Dropout(0.3)
        )


    def forward(self,x):

        return self.encoder(x)



class MultimodalFusionClassifier(nn.Module):

    def __init__(
        self,
        tabular_dim=40,
        text_dim=768,
        graph_dim=3
    ):

        super().__init__()


        self.tabular_encoder = FeatureEncoder(
            tabular_dim,
            64
        )


        self.text_encoder = FeatureEncoder(
            text_dim,
            128
        )


        self.graph_encoder = FeatureEncoder(
            graph_dim,
            32
        )


        self.classifier = nn.Sequential(
            nn.Linear(
                224,
                64
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                64,
                2
            )
        )


    def forward(
        self,
        tabular,
        text,
        graph
    ):

        t = self.tabular_encoder(tabular)

        x = self.text_encoder(text)

        g = self.graph_encoder(graph)


        fused = torch.cat(
            [
                t,
                x,
                g
            ],
            dim=1
        )


        return self.classifier(fused)

In [45]:
# ============================================================
# CELL 29
# Load trained multimodal fusion model
# ============================================================


from pathlib import Path
import torch



print("="*90)
print("LOAD MULTIMODAL FUSION CHECKPOINT")
print("="*90)



MODEL_DIR = (
    BASE_DIR /
    "models"
)


MODEL_PATH = (
    MODEL_DIR /
    "multimodal_fusion_best.pt"
)



device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)



print(
    "Device:",
    device
)



# initialize model

model = MultimodalFusionClassifier(
    tabular_dim=40,
    text_dim=768,
    graph_dim=3
)



# load weights

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)



print(
    "Checkpoint type:",
    type(checkpoint)
)



# handle different save formats

if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

    else:

        model.load_state_dict(
            checkpoint
        )

else:

    model.load_state_dict(
        checkpoint
    )



model.to(device)


model.eval()



print()
print("Model loaded successfully")


LOAD MULTIMODAL FUSION CHECKPOINT
Device: cpu
Checkpoint type: <class 'collections.OrderedDict'>

Model loaded successfully


In [46]:
# ============================================================
# CELL 30
# Run inference on unlabeled users
# ============================================================


print("=" * 90)
print("UNLABELED INFERENCE")
print("=" * 90)


import torch
import numpy as np
from tqdm.auto import tqdm



all_logits = []
all_probs = []
all_keys = []



model.eval()


with torch.no_grad():

    for batch in tqdm(
        unlabeled_loader
    ):

        tabular = (
            batch["tabular"]
            .to(device)
        )

        text = (
            batch["text"]
            .to(device)
        )

        graph = (
            batch["graph"]
            .to(device)
        )


        logits = model(
            tabular,
            text,
            graph
        )


        probs = torch.softmax(
            logits,
            dim=1
        )


        all_logits.append(
            logits.cpu().numpy()
        )


        all_probs.append(
            probs.cpu().numpy()
        )


        all_keys.extend(
            batch["user_key"]
        )



all_logits = np.vstack(
    all_logits
)


all_probs = np.vstack(
    all_probs
)



print(
    "Logits shape:",
    all_logits.shape
)


print(
    "Probabilities shape:",
    all_probs.shape
)


print(
    "Users:",
    len(all_keys)
)

UNLABELED INFERENCE


  0%|          | 0/287 [00:00<?, ?it/s]

Logits shape: (18331, 2)
Probabilities shape: (18331, 2)
Users: 18331


In [48]:
# ============================================================
# CELL 31
# Create Pseudo Prediction DataFrame
# ============================================================


import pandas as pd
import numpy as np



print("=" * 90)
print("CREATE PREDICTION DATAFRAME")
print("=" * 90)



pseudo_df = pd.DataFrame({

    "user_key":
        all_keys,


    "prob_class_0":
        all_probs[:,0],


    "prob_class_1":
        all_probs[:,1]

})



# predicted class

pseudo_df["predicted_label"] = (
    np.argmax(
        all_probs,
        axis=1
    )
)



# confidence

pseudo_df["confidence"] = (
    np.max(
        all_probs,
        axis=1
    )
)



print(
    pseudo_df.shape
)


pseudo_df.head()

CREATE PREDICTION DATAFRAME
(18331, 5)


,user_key,prob_class_0,prob_class_1,predicted_label,confidence
0,patriotpointman,0.090056,0.909944,1,0.909944
1,iranziba1401,0.999734,0.000266,0,0.999734
2,miiran20194,0.999622,0.000378,0,0.999622
3,artemis540721,0.999998,0.000002,0,0.999998
4,tifraghe_n,0.998180,0.001820,0,0.998180


In [56]:
# ============================================================
# CELL 33
# Generate Pseudo Labels
# ============================================================


print("=" * 90)
print("GENERATE PSEUDO LABELS")
print("=" * 90)


import pandas as pd
import numpy as np



pseudo_labels_df = pd.DataFrame({

    "user_key":
        all_keys,


    "prob_human":
        all_probs[:, 0],


    "prob_bot":
        all_probs[:, 1]

})



# model prediction

pseudo_labels_df["pseudo_label"] = (
    np.argmax(
        all_probs,
        axis=1
    )
)



# confidence

pseudo_labels_df["confidence"] = (
    np.max(
        all_probs,
        axis=1
    )
)



print(
    "Shape:",
    pseudo_labels_df.shape
)



print(
    "\nPseudo label distribution:"
)


print(
    pseudo_labels_df[
        "pseudo_label"
    ]
    .value_counts()
)



pseudo_labels_df.head()

GENERATE PSEUDO LABELS
Shape: (18331, 5)

Pseudo label distribution:
pseudo_label
0    15695
1     2636
Name: count, dtype: int64


,user_key,prob_human,prob_bot,pseudo_label,confidence
0,patriotpointman,0.090056,0.909944,1,0.909944
1,iranziba1401,0.999734,0.000266,0,0.999734
2,miiran20194,0.999622,0.000378,0,0.999622
3,artemis540721,0.999998,0.000002,0,0.999998
4,tifraghe_n,0.998180,0.001820,0,0.998180


In [96]:
# ============================================================
# CELL 34
# Confidence Analysis
# ============================================================


print("=" * 90)
print("CONFIDENCE ANALYSIS")
print("=" * 90)



print(
    pseudo_labels_df["confidence"]
    .describe()
)



print("\nThreshold analysis:")


for threshold in [
    0.80,
    0.85,
    0.90,
    0.95,
    0.99,
    0.999,
    0.999999,
    0.9999999999,
    0.99999999999999
]:

    count = (
        pseudo_labels_df["confidence"]
        >= threshold
    ).sum()


    ratio = (
        count /
        len(pseudo_labels_df)
        * 100
    )


    print(
        f"Confidence >= {threshold}: "
        f"{count} users ({ratio:.2f}%)"
    )

CONFIDENCE ANALYSIS
count    18331.000000
mean         0.969362
std          0.087849
min          0.500327
25%          0.997700
50%          0.999999
75%          1.000000
max          1.000000
Name: confidence, dtype: float64

Threshold analysis:
Confidence >= 0.8: 17193 users (93.79%)
Confidence >= 0.85: 16914 users (92.27%)
Confidence >= 0.9: 16554 users (90.31%)
Confidence >= 0.95: 15940 users (86.96%)
Confidence >= 0.99: 14719 users (80.30%)
Confidence >= 0.999: 13208 users (72.05%)
Confidence >= 0.999999: 9212 users (50.25%)
Confidence >= 0.9999999999: 7734 users (42.19%)
Confidence >= 0.99999999999999: 7734 users (42.19%)


In [213]:
# ============================================================
# CELL 35
# Confidence Filtering
# ============================================================


CONFIDENCE_THRESHOLD = 0.999999


pseudo_filtered_df = (
    pseudo_labels_df[
        pseudo_labels_df["confidence"]
        >= CONFIDENCE_THRESHOLD
    ]
    .copy()
)



print("=" * 90)
print("CONFIDENCE FILTERING")
print("=" * 90)


print(
    "Threshold:",
    CONFIDENCE_THRESHOLD
)


print(
    "Selected users:",
    len(pseudo_filtered_df)
)


print(
    "Percentage:",
    len(pseudo_filtered_df)
    /
    len(pseudo_labels_df)
    *
    100
)



print("\nPseudo label distribution:")


print(
    pseudo_filtered_df[
        "pseudo_label"
    ]
    .value_counts()
)

CONFIDENCE FILTERING
Threshold: 0.999999
Selected users: 9212
Percentage: 50.25366864873712

Pseudo label distribution:
pseudo_label
0    9139
1      73
Name: count, dtype: int64


In [214]:
# ============================================================
# CELL 36
# Pseudo Label Modality Analysis
# ============================================================


print("=" * 90)
print("PSEUDO LABEL MODALITY ANALYSIS")
print("=" * 90)



pseudo_analysis_df = (
    pseudo_filtered_df
    .merge(
        true_unlabeled[
            [
                "user_key",
                "has_description",
                "has_graph",
                "has_behavior_temporal"
            ]
        ],
        on="user_key",
        how="left"
    )
)



print("\nText availability:")

print(
    pseudo_analysis_df[
        "has_description"
    ]
    .value_counts()
)



print("\nGraph availability:")

print(
    pseudo_analysis_df[
        "has_graph"
    ]
    .value_counts()
)



print("\nTemporal availability:")

print(
    pseudo_analysis_df[
        "has_behavior_temporal"
    ]
    .value_counts()
)

PSEUDO LABEL MODALITY ANALYSIS

Text availability:
has_description
1    5492
0    3720
Name: count, dtype: int64

Graph availability:
has_graph
0    7672
1    1540
Name: count, dtype: int64

Temporal availability:
has_behavior_temporal
1    9212
Name: count, dtype: int64


In [215]:
# ============================================================
# CELL 37
# SAVE PSEUDO LABELS
# ============================================================


from pathlib import Path


print("=" * 90)
print("SAVE PSEUDO LABELS")
print("=" * 90)



# ------------------------------------------------------------
# Semi-supervised output directory
# ------------------------------------------------------------

SEMI_SUPERVISED_DIR = (
    BASE_DIR /
    "final_data" /
    "semi_supervised"
)


SEMI_SUPERVISED_DIR.mkdir(
    parents=True,
    exist_ok=True
)



# ------------------------------------------------------------
# Save path
# ------------------------------------------------------------

PSEUDO_LABEL_PATH = (
    SEMI_SUPERVISED_DIR /
    "pseudo_labels_18331.csv"
)



# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

pseudo_filtered_df.to_csv(
    PSEUDO_LABEL_PATH,
    index=False
)



print(
    "Saved:"
)

print(
    PSEUDO_LABEL_PATH
)



print(
    "\nShape:"
)

print(
    pseudo_filtered_df.shape
)



print(
    "\nColumns:"
)

print(
    pseudo_filtered_df.columns.tolist()
)

SAVE PSEUDO LABELS
Saved:
..\final_data\semi_supervised\pseudo_labels_18331.csv

Shape:
(9212, 5)

Columns:
['user_key', 'prob_human', 'prob_bot', 'pseudo_label', 'confidence']


In [216]:
# ============================================================
# CELL 38
# BUILD SEMI-SUPERVISED TRAINING METADATA
# ============================================================


print("=" * 90)
print("BUILD SEMI-SUPERVISED TRAINING METADATA")
print("=" * 90)


import pandas as pd



# ------------------------------------------------------------
# Real labeled samples
# ------------------------------------------------------------

labeled_meta = labeled_df[
    [
        "user_key",
        "label_binary"
    ]
].copy()



labeled_meta = labeled_meta.rename(
    columns={
        "label_binary": "label"
    }
)



labeled_meta["label_source"] = (
    "human_label"
)



labeled_meta["sample_weight"] = (
    1.0
)



labeled_meta["confidence"] = (
    1.0
)



# ------------------------------------------------------------
# Pseudo labeled samples
# ------------------------------------------------------------

pseudo_meta = pseudo_filtered_df[
    [
        "user_key",
        "pseudo_label",
        "confidence"
    ]
].copy()



pseudo_meta = pseudo_meta.rename(
    columns={
        "pseudo_label": "label"
    }
)



pseudo_meta["label_source"] = (
    "pseudo_label"
)



# initial pseudo weight

pseudo_meta["sample_weight"] = (
    0.5
)



# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

semi_supervised_meta = pd.concat(
    [
        labeled_meta,
        pseudo_meta
    ],
    ignore_index=True
)



print(
    "Total samples:",
    len(semi_supervised_meta)
)



print(
    "\nLabel distribution:"
)

print(
    semi_supervised_meta[
        "label"
    ].value_counts()
)



print(
    "\nSource distribution:"
)

print(
    semi_supervised_meta[
        "label_source"
    ].value_counts()
)

BUILD SEMI-SUPERVISED TRAINING METADATA
Total samples: 10171

Label distribution:
label
0    9911
1     260
Name: count, dtype: int64

Source distribution:
label_source
pseudo_label    9212
human_label      959
Name: count, dtype: int64


In [217]:
# ============================================================
# CELL 39
# LOAD LABELED MULTIMODAL FEATURES
# ============================================================


print("=" * 90)
print("LOAD LABELED FEATURES")
print("=" * 90)


import pandas as pd
import numpy as np



LABELED_MODEL_PATH = (
    BASE_DIR /
    "final_data" /
    "users" /
    "modeling_labeled_binary.csv"
)



labeled_features_df = pd.read_csv(
    LABELED_MODEL_PATH
)



print(
    "Shape:",
    labeled_features_df.shape
)


print(
    labeled_features_df[
        [
            "user_key",
            "label_binary"
        ]
    ].head()
)

LOAD LABELED FEATURES
Shape: (959, 56)
        user_key  label_binary
0   kathymobarez             1
1   rahe_azadi_5             1
2      dadkhahim             1
3     hiwatubeai             1
4  kazeroonkings             1


In [218]:
# ============================================================
# CELL 40
# LOAD LABELED TEXT EMBEDDINGS
# ============================================================


print("=" * 90)
print("LOAD LABELED TEXT EMBEDDINGS")
print("=" * 90)


import numpy as np
from pathlib import Path



EMBED_DIR = (
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr"
)



train_emb_path = (
    EMBED_DIR /
    "train_user_embeddings.npy"
)

val_emb_path = (
    EMBED_DIR /
    "val_user_embeddings.npy"
)

test_emb_path = (
    EMBED_DIR /
    "test_user_embeddings.npy"
)



train_embeddings = np.load(
    train_emb_path
)

val_embeddings = np.load(
    val_emb_path
)

test_embeddings = np.load(
    test_emb_path
)



print("Train:", train_embeddings.shape)
print("Val:", val_embeddings.shape)
print("Test:", test_embeddings.shape)

LOAD LABELED TEXT EMBEDDINGS
Train: (671, 768)
Val: (144, 768)
Test: (143, 768)


In [219]:
# ============================================================
# CELL 41
# CHECK LABELED TEXT ALIGNMENT (FIXED)
# ============================================================


print("=" * 90)
print("CHECK LABELED TEXT ALIGNMENT")
print("=" * 90)


import numpy as np



EMBED_DIR = (
    BASE_DIR /
    "final_data" /
    "embeddings" /
    "xlmr"
)



train_keys = np.load(
    EMBED_DIR /
    "train_user_keys.npy",
    allow_pickle=True
)

val_keys = np.load(
    EMBED_DIR /
    "val_user_keys.npy",
    allow_pickle=True
)

test_keys = np.load(
    EMBED_DIR /
    "test_user_keys.npy",
    allow_pickle=True
)



print("Raw train keys shape:", train_keys.shape)
print("Raw val keys shape:", val_keys.shape)
print("Raw test keys shape:", test_keys.shape)



# flatten if nested

train_keys = np.array(
    train_keys
).reshape(-1)


val_keys = np.array(
    val_keys
).reshape(-1)


test_keys = np.array(
    test_keys
).reshape(-1)



all_text_keys = np.concatenate(
    [
        train_keys,
        val_keys,
        test_keys
    ]
)



print("\nAfter flatten:")

print(
    "Embedding users:",
    len(all_text_keys)
)


print(
    "Unique users:",
    len(set(all_text_keys))
)


print(
    "Labeled csv users:",
    labeled_features_df["user_key"].nunique()
)



missing_users = set(
    labeled_features_df["user_key"]
) - set(
    all_text_keys
)



print(
    "\nMissing text embeddings:"
)

print(
    len(missing_users)
)


print(
    list(missing_users)[:10]
)

CHECK LABELED TEXT ALIGNMENT
Raw train keys shape: (671, 1)
Raw val keys shape: (144, 1)
Raw test keys shape: (143, 1)

After flatten:
Embedding users: 958
Unique users: 958
Labeled csv users: 959

Missing text embeddings:
1
['mostafamehraeen']


In [220]:
# ============================================================
# CELL 42
# BUILD LABELED TEXT EMBEDDING MATRIX
# ============================================================


print("=" * 90)
print("BUILD LABELED TEXT EMBEDDINGS")
print("=" * 90)


import numpy as np



# ------------------------------------------------------------
# Combine embeddings
# ------------------------------------------------------------

labeled_embeddings = np.concatenate(
    [
        train_embeddings,
        val_embeddings,
        test_embeddings
    ],
    axis=0
)



labeled_keys = np.concatenate(
    [
        train_keys,
        val_keys,
        test_keys
    ],
    axis=0
)



# ------------------------------------------------------------
# Create lookup
# ------------------------------------------------------------

text_embedding_lookup = {
    key: emb
    for key, emb in zip(
        labeled_keys,
        labeled_embeddings
    )
}



TEXT_DIM = 768



final_labeled_text_embeddings = []



missing_text_count = 0



for user_key in labeled_features_df["user_key"]:

    if user_key in text_embedding_lookup:

        final_labeled_text_embeddings.append(
            text_embedding_lookup[user_key]
        )

    else:

        final_labeled_text_embeddings.append(
            np.zeros(
                TEXT_DIM,
                dtype=np.float32
            )
        )

        missing_text_count += 1



final_labeled_text_embeddings = np.vstack(
    final_labeled_text_embeddings
)



print(
    "Final shape:",
    final_labeled_text_embeddings.shape
)



print(
    "Missing text embeddings replaced:",
    missing_text_count
)

BUILD LABELED TEXT EMBEDDINGS
Final shape: (959, 768)
Missing text embeddings replaced: 1


In [221]:
# ============================================================
# CELL 43-A
# LOAD FEATURE POLICY
# ============================================================

print("=" * 90)
print("LOAD FEATURE POLICY")
print("=" * 90)


from pathlib import Path
import json



CONFIG_DIR = (
    BASE_DIR /
    "final_data" /
    "config"
)



MODEL_FEATURE_POLICY_PATH = (
    CONFIG_DIR /
    "model_feature_policy.json"
)



with open(
    MODEL_FEATURE_POLICY_PATH,
    "r",
    encoding="utf-8"
) as f:
    
    model_feature_policy = json.load(f)



print(
    "Policy loaded:"
)

print(
    MODEL_FEATURE_POLICY_PATH
)



print(
    "\nTop-level keys:"
)


print(
    model_feature_policy.keys()
)

LOAD FEATURE POLICY
Policy loaded:
..\final_data\config\model_feature_policy.json

Top-level keys:
dict_keys(['target', 'final_model_tabular_features', 'numeric_features', 'binary_features', 'categorical_features', 'text_fields', 'modality_masks', 'removed_constant_or_unusable_features', 'external_detector_features_excluded', 'annotation_columns_excluded', 'n_final_tabular_features'])


In [222]:
# ============================================================
# CELL 43-B
# VALIDATE FEATURE POLICY
# ============================================================


print("=" * 90)
print("VALIDATE FEATURE POLICY")
print("=" * 90)


numeric_features = (
    model_feature_policy["numeric_features"]
)


binary_features = (
    model_feature_policy["binary_features"]
)


final_features = (
    model_feature_policy["final_model_tabular_features"]
)



print(
    "Numeric features:",
    len(numeric_features)
)


print(
    "Binary features:",
    len(binary_features)
)


print(
    "Final tabular features:",
    len(final_features)
)


print(
    "\nExpected:",
    model_feature_policy["n_final_tabular_features"]
)


VALIDATE FEATURE POLICY
Numeric features: 36
Binary features: 6
Final tabular features: 42

Expected: 42


In [223]:
# ============================================================
# CELL 43-C
# BUILD LABELED TABULAR FEATURES (ARTIFACT DRIVEN)
# ============================================================


print("=" * 90)
print("BUILD LABELED TABULAR FEATURES")
print("=" * 90)


import numpy as np
import joblib



# ------------------------------------------------------------
# Load feature groups from policy
# ------------------------------------------------------------

numeric_features = (
    model_feature_policy["numeric_features"]
)


binary_features = (
    model_feature_policy["binary_features"]
)



# ------------------------------------------------------------
# Load preprocessing artifacts
# ------------------------------------------------------------

PREPROCESS_DIR = (
    BASE_DIR /
    "models" /
    "preprocessing"
)



imputer = joblib.load(
    PREPROCESS_DIR /
    "numeric_median_imputer.joblib"
)


scaler = joblib.load(
    PREPROCESS_DIR /
    "robust_scaler.joblib"
)



print(
    "Policy numeric features:",
    len(numeric_features)
)


print(
    "Scaler features:",
    len(scaler.feature_names_in_)
)



# ------------------------------------------------------------
# Prepare numeric dataframe
# ------------------------------------------------------------

labeled_numeric_df = (
    labeled_features_df[
        numeric_features
    ]
    .copy()
)



# ------------------------------------------------------------
# Imputation
# ------------------------------------------------------------

labeled_numeric_array = (
    imputer.transform(
        labeled_numeric_df
    )
)



# Back to dataframe using policy names
# to allow scaler feature selection

import pandas as pd


labeled_numeric_df = pd.DataFrame(
    labeled_numeric_array,
    columns=numeric_features
)



# ------------------------------------------------------------
# Select exactly scaler features
# ------------------------------------------------------------

scaler_features = (
    list(
        scaler.feature_names_in_
    )
)



labeled_scaled = scaler.transform(
    labeled_numeric_df[
        scaler_features
    ]
)



# ------------------------------------------------------------
# Binary features
# ------------------------------------------------------------

labeled_binary = (
    labeled_features_df[
        binary_features
    ]
    .fillna(0)
    .astype(float)
    .values
)



# ------------------------------------------------------------
# Final tabular vector
# ------------------------------------------------------------

final_labeled_tabular = np.concatenate(
    [
        labeled_scaled,
        labeled_binary
    ],
    axis=1
)



print("\nFinal shape:")
print(
    final_labeled_tabular.shape
)



print("\nExpected:")
print(
    model_feature_policy[
        "n_final_tabular_features"
    ]
)



print(
    "\nNaN:",
    np.isnan(final_labeled_tabular).sum()
)


print(
    "Inf:",
    np.isinf(final_labeled_tabular).sum()
)

BUILD LABELED TABULAR FEATURES
Policy numeric features: 36
Scaler features: 33

Final shape:
(959, 39)

Expected:
42

NaN: 0
Inf: 0


In [224]:
# ============================================================
# CELL 43-C
# BUILD LABELED TABULAR FEATURES (FINAL)
# ============================================================


print("=" * 90)
print("BUILD LABELED TABULAR FEATURES")
print("=" * 90)


import numpy as np
import pandas as pd
import joblib



# ------------------------------------------------------------
# Feature groups from artifacts
# ------------------------------------------------------------

numeric_features = (
    model_feature_policy["numeric_features"]
)


binary_features = (
    model_feature_policy["binary_features"]
)



scaler_features = list(
    scaler.feature_names_in_
)



unscaled_numeric_features = list(
    set(numeric_features)
    -
    set(scaler_features)
)



print(
    "Scaled numeric:",
    len(scaler_features)
)


print(
    "Unscaled numeric:",
    len(unscaled_numeric_features)
)


print(
    "Binary:",
    len(binary_features)
)



# ------------------------------------------------------------
# Numeric dataframe
# ------------------------------------------------------------

numeric_df = (
    labeled_features_df[
        numeric_features
    ]
    .copy()
)



# ------------------------------------------------------------
# Imputation
# ------------------------------------------------------------

numeric_array = imputer.transform(
    numeric_df
)


numeric_df = pd.DataFrame(
    numeric_array,
    columns=numeric_features
)



# ------------------------------------------------------------
# Scaled numeric
# ------------------------------------------------------------

scaled_numeric = scaler.transform(
    numeric_df[
        scaler_features
    ]
)



# ------------------------------------------------------------
# Unscaled numeric
# ------------------------------------------------------------

unscaled_numeric = (
    numeric_df[
        unscaled_numeric_features
    ]
    .values
)



# ------------------------------------------------------------
# Binary
# ------------------------------------------------------------

binary = (
    labeled_features_df[
        binary_features
    ]
    .fillna(0)
    .astype(float)
    .values
)



# ------------------------------------------------------------
# Final concat
# ------------------------------------------------------------

final_labeled_tabular = np.concatenate(
    [
        scaled_numeric,
        unscaled_numeric,
        binary
    ],
    axis=1
)



print("\nFinal shape:")
print(
    final_labeled_tabular.shape
)


print("\nExpected:")
print(
    model_feature_policy[
        "n_final_tabular_features"
    ]
)


print(
    "\nNaN:",
    np.isnan(final_labeled_tabular).sum()
)


print(
    "Inf:",
    np.isinf(final_labeled_tabular).sum()
)

BUILD LABELED TABULAR FEATURES
Scaled numeric: 33
Unscaled numeric: 3
Binary: 6

Final shape:
(959, 42)

Expected:
42

NaN: 0
Inf: 0


In [225]:
# ============================================================
# CELL 44
# LOAD GRAPH STATISTICS
# ============================================================


print("=" * 90)
print("LOAD GRAPH STATISTICS")
print("=" * 90)


import pandas as pd
import numpy as np



GRAPH_DIR = (
    BASE_DIR /
    "final_data" /
    "graph"
)



GRAPH_STATS_PATH = (
    GRAPH_DIR /
    "graph_user_statistics.csv"
)



graph_stats_df = pd.read_csv(
    GRAPH_STATS_PATH
)



print(
    "Shape:",
    graph_stats_df.shape
)



print(
    "\nColumns:"
)


print(
    graph_stats_df.columns.tolist()
)



graph_stats_df.head()

LOAD GRAPH STATISTICS
Shape: (3443, 8)

Columns:
['id', 'screen_name', 'followers', 'following', 'followers_list', 'following_list', 'collected_followers_count', 'collected_following_count']


,id,screen_name,followers,following,followers_list,following_list,collected_followers_count,collected_following_count
0,1.780000e+18,SajjadSade48567,"chaiiee, dr_lizzii, nathaniel2026, julian270zi...","grok, USABehFarsi, Psiphon_Fa, PersianDJT, Pho...","['chaiiee', 'dr_lizzii', 'nathaniel2026', 'jul...","['grok', 'USABehFarsi', 'Psiphon_Fa', 'Persian...",7,42
1,2.511595e+08,mojtaba2a,"aryan78093797, BanihashemiNav1, Riseagain979, ...","behrouzina, twiterrism819, monikaa2500, aryame...","['aryan78093797', 'BanihashemiNav1', 'Riseagai...","['behrouzina', 'twiterrism819', 'monikaa2500',...",193,160
2,1.790000e+18,niya64_,"Mentttaallll, TalbB909091, santinokarimi, Java...","chizbiin, Alireza774365, zahraaa1988, nooshiin...","['Mentttaallll', 'TalbB909091', 'santinokarimi...","['chizbiin', 'Alireza774365', 'zahraaa1988', '...",206,189
3,1.870000e+18,arash_the3rd,"Avayiran, darkpixele, sjavidnia, AnahitaSarab,...","pirozzzzzzzzz, Earendilist, salargholamiii, si...","['Avayiran', 'darkpixele', 'sjavidnia', 'Anahi...","['pirozzzzzzzzz', 'Earendilist', 'salargholami...",179,162
4,1.520000e+18,kaveAhanga2022,"leovirgo_17_, yutaabm, ShahdadmmMajid, Aliasad...","FanpageYaspah, leovirgo_17_, seyedhadikasaei, ...","['leovirgo_17_', 'yutaabm', 'ShahdadmmMajid', ...","['FanpageYaspah', 'leovirgo_17_', 'seyedhadika...",166,111


In [226]:
# ============================================================
# CHECK GRAPH VARIABLES
# ============================================================


print("=" * 90)
print("AVAILABLE GRAPH VARIABLES")
print("=" * 90)


graph_vars = [
    name
    for name in globals().keys()
    if "graph" in name.lower()
]


for v in graph_vars:
    print(v)

AVAILABLE GRAPH VARIABLES
GRAPH_DIR
graph_keys
graph_node_ids
graph_features_unlabeled
graph
GRAPH_STATS_PATH
graph_stats_df
graph_vars
GRAPH_DIM
graph_feature_lookup
final_labeled_graph
missing_graph_count
graph_lookup
pseudo_graph
semi_graph


In [227]:
# ============================================================
# CELL 44-A (FIX)
# INSPECT GRAPH OBJECTS
# ============================================================


print("=" * 90)
print("INSPECT GRAPH OBJECTS")
print("=" * 90)



for name in [
    "graph_keys",
    "graph_node_ids",
    "graph_features_unlabeled"
]:

    obj = globals()[name]

    print("\n", name)
    print("-" * 40)

    print(
        "type:",
        type(obj)
    )


    if hasattr(obj, "shape"):

        print(
            "shape:",
            obj.shape
        )


    if hasattr(obj, "__len__"):

        print(
            "length:",
            len(obj)
        )



print("\nFirst graph node ids:")

print(
    list(graph_node_ids)[:5]
)



print("\nFirst graph features:")

print(
    graph_features_unlabeled[:5]
)

INSPECT GRAPH OBJECTS

 graph_keys
----------------------------------------
type: <class 'set'>
length: 13465

 graph_node_ids
----------------------------------------
type: <class 'list'>
length: 18331

 graph_features_unlabeled
----------------------------------------
type: <class 'numpy.ndarray'>
shape: (18331, 3)
length: 18331

First graph node ids:
[-1, -1, 4005, 2463, 12771]

First graph features:
[[ 0.  0.  0.]
 [ 0.  0.  0.]
 [33. 33. 66.]
 [31. 31. 62.]
 [ 1.  1.  2.]]


In [228]:
# ============================================================
# CELL 44-B
# BUILD LABELED GRAPH FEATURES
# ============================================================


print("=" * 90)
print("BUILD LABELED GRAPH FEATURES")
print("=" * 90)


import numpy as np



GRAPH_DIM = (
    graph_features_unlabeled.shape[1]
)



# ------------------------------------------------------------
# Build graph lookup
# ------------------------------------------------------------

graph_feature_lookup = {}



for user_key, feat in zip(
    true_unlabeled["user_key"],
    graph_features_unlabeled
):

    graph_feature_lookup[user_key] = feat



# ------------------------------------------------------------
# Generate labeled graph matrix
# ------------------------------------------------------------

final_labeled_graph = []

missing_graph_count = 0



for user_key in labeled_features_df["user_key"]:

    if user_key in graph_feature_lookup:

        final_labeled_graph.append(
            graph_feature_lookup[user_key]
        )

    else:

        final_labeled_graph.append(
            np.zeros(
                GRAPH_DIM,
                dtype=np.float32
            )
        )

        missing_graph_count += 1



final_labeled_graph = np.vstack(
    final_labeled_graph
)



print(
    "Final shape:"
)

print(
    final_labeled_graph.shape
)



print(
    "\nGraph dimension:"
)

print(
    GRAPH_DIM
)



print(
    "\nMissing graph replaced:"
)

print(
    missing_graph_count
)



print(
    "\nUsers with graph:"
)

print(
    len(labeled_features_df)
    -
    missing_graph_count
)

BUILD LABELED GRAPH FEATURES
Final shape:
(959, 3)

Graph dimension:
3

Missing graph replaced:
959

Users with graph:
0


In [229]:
# ============================================================
# CELL 44-C
# CHECK GRAPH KEY MATCHING
# ============================================================


print("=" * 90)
print("CHECK GRAPH KEY MATCHING")
print("=" * 90)



labeled_keys = set(
    labeled_features_df["user_key"]
)



matched = (
    labeled_keys
    &
    graph_keys
)



print(
    "Labeled users:",
    len(labeled_keys)
)



print(
    "Graph keys:",
    len(graph_keys)
)



print(
    "Matched:",
    len(matched)
)



print(
    "\nExamples:"
)

print(
    list(matched)[:10]
)

CHECK GRAPH KEY MATCHING
Labeled users: 959
Graph keys: 13465
Matched: 935

Examples:
['zhina2019', 'obeydzacany', 'kouli2017', 'parpanchi', 'iraj63569385', 'firstzorba1', 'danamehraban', 'zhiyan_shah', 'ansari_bahman', 'rezarez11109331']


In [230]:
# ============================================================
# CELL 44-D
# BUILD LABELED GRAPH FEATURES (FINAL)
# ============================================================


print("=" * 90)
print("BUILD LABELED GRAPH FEATURES")
print("=" * 90)



import numpy as np



GRAPH_DIM = graph_features_unlabeled.shape[1]



# ------------------------------------------------------------
# Build user_key -> graph feature lookup
# ------------------------------------------------------------

graph_lookup = {}



for idx, user_key in enumerate(
    true_unlabeled["user_key"]
):

    node_id = graph_node_ids[idx]

    if node_id != -1:

        graph_lookup[user_key] = (
            graph_features_unlabeled[idx]
        )



print(
    "Graph lookup size:",
    len(graph_lookup)
)



# ------------------------------------------------------------
# Create labeled graph matrix
# ------------------------------------------------------------

final_labeled_graph = []


missing_graph_count = 0



for user_key in labeled_features_df["user_key"]:


    if user_key in graph_lookup:


        final_labeled_graph.append(
            graph_lookup[user_key]
        )


    else:


        final_labeled_graph.append(
            np.zeros(
                GRAPH_DIM,
                dtype=np.float32
            )
        )

        missing_graph_count += 1



final_labeled_graph = np.vstack(
    final_labeled_graph
)



print("\nFinal shape:")

print(
    final_labeled_graph.shape
)



print(
    "\nMissing graph replaced:"
)

print(
    missing_graph_count
)



print(
    "\nUsers with graph:"
)

print(
    len(labeled_features_df)
    -
    missing_graph_count
)

BUILD LABELED GRAPH FEATURES
Graph lookup size: 12397

Final shape:
(959, 3)

Missing graph replaced:
959

Users with graph:
0


In [231]:
# ============================================================
# CHECK GRAPH MAPPING VARIABLES (FIXED)
# ============================================================


print("=" * 90)
print("CHECK GRAPH MAPPING VARIABLES")
print("=" * 90)


for name in list(globals().keys()):

    value = globals()[name]

    name_lower = name.lower()

    if (
        "node" in name_lower
        or "user" in name_lower
        or "map" in name_lower
        or "key" in name_lower
    ):

        if name not in [
            "labeled_features_df",
            "true_unlabeled"
        ]:

            print(
                name,
                "|",
                type(value)
            )

CHECK GRAPH MAPPING VARIABLES
node_to_id | <class 'dict'>
user_keys | <class 'pandas.Series'>
graph_keys | <class 'set'>
matched_nodes | <class 'numpy.ndarray'>
key | <class 'str'>
normalized_node_to_id | <class 'dict'>
graph_node_ids | <class 'list'>
user_key | <class 'str'>
node_id | <class 'int'>
train_user_emb | <class 'numpy.ndarray'>
train_user_keys | <class 'numpy.ndarray'>
all_keys | <class 'list'>
train_keys | <class 'numpy.ndarray'>
val_keys | <class 'numpy.ndarray'>
test_keys | <class 'numpy.ndarray'>
all_text_keys | <class 'numpy.ndarray'>
missing_users | <class 'set'>
labeled_keys | <class 'set'>
pseudo_user_keys | <class 'pandas.arrays.StringArray'>
semi_user_keys | <class 'numpy.ndarray'>


In [232]:
# ============================================================
# BUILD LABELED GRAPH FEATURES FINAL
# ============================================================


import numpy as np


GRAPH_DIM = graph_features_unlabeled.shape[1]


final_labeled_graph = []


missing_graph_count = 0



for user_key in labeled_features_df["user_key"]:


    if user_key in normalized_node_to_id:


        node_id = normalized_node_to_id[user_key]


        final_labeled_graph.append(
            graph_features_unlabeled[node_id]
        )


    else:


        final_labeled_graph.append(
            np.zeros(
                GRAPH_DIM,
                dtype=np.float32
            )
        )

        missing_graph_count += 1



final_labeled_graph = np.vstack(
    final_labeled_graph
)



print("=" * 90)

print(
    "Final shape:",
    final_labeled_graph.shape
)


print(
    "Missing graph:",
    missing_graph_count
)


print(
    "Available graph:",
    len(labeled_features_df)-missing_graph_count
)

Final shape: (959, 3)
Missing graph: 24
Available graph: 935


In [233]:
# ============================================================
# CELL 45
# VALIDATE PSEUDO MODALITY SHAPES
# ============================================================


print("=" * 90)
print("VALIDATE PSEUDO MODALITY SHAPES")
print("=" * 90)



print(
    "Unlabeled Tabular:"
)

print(
    final_tabular_df.shape
)



print(
    "\nUnlabeled Text:"
)

print(
    unlabeled_text_embeddings.shape
)



print(
    "\nUnlabeled Graph:"
)

print(
    graph_features_unlabeled.shape
)



print(
    "\nPseudo labels:"
)

print(
    pseudo_labels_df.shape
)

VALIDATE PSEUDO MODALITY SHAPES
Unlabeled Tabular:
(18331, 40)

Unlabeled Text:
(18331, 768)

Unlabeled Graph:
(18331, 3)

Pseudo labels:
(18331, 5)


In [298]:
# ============================================================
# ALIGN PSEUDO MODALITIES
# ============================================================

print("=" * 90)
print("ALIGN PSEUDO MODALITIES")
print("=" * 90)


import numpy as np


# pseudo users in correct order

pseudo_user_keys = (
    pseudo_labels_filtered_df["user_key"]
    .values
)


# build lookup

unlabeled_index = {
    key: idx
    for idx, key in enumerate(
        true_unlabeled["user_key"]
    )
}


pseudo_indices = np.array(
    [
        unlabeled_index[key]
        for key in pseudo_user_keys
    ]
)



# select modalities

pseudo_tabular = (
    final_tabular_df.values[
        pseudo_indices
    ]
)


pseudo_text = (
    unlabeled_text_embeddings[
        pseudo_indices
    ]
)


pseudo_graph = (
    graph_features_unlabeled[
        pseudo_indices
    ]
)



print(
    "Pseudo tabular:",
    pseudo_tabular.shape
)


print(
    "Pseudo text:",
    pseudo_text.shape
)


print(
    "Pseudo graph:",
    pseudo_graph.shape
)

ALIGN PSEUDO MODALITIES
Pseudo tabular: (9212, 40)
Pseudo text: (9212, 768)
Pseudo graph: (9212, 3)


In [300]:
# ============================================================
# BUILD FINAL SEMI-SUPERVISED ARRAYS
# ============================================================

print("=" * 90)
print("BUILD FINAL SEMI-SUPERVISED ARRAYS")
print("=" * 90)


import numpy as np



# ------------------------------------------------------------
# Human labeled data
# ------------------------------------------------------------

human_tabular = (
    final_labeled_features
)


human_text = (
    final_labeled_text_embeddings
)


human_graph = (
    final_labeled_graph_features
)


human_labels = (
    labeled_features_df["label_binary"]
    .values
)


human_weights = np.ones(
    len(human_labels),
    dtype=np.float32
)



# ------------------------------------------------------------
# Pseudo labeled data
# ------------------------------------------------------------

pseudo_labels = (
    pseudo_labels_filtered_df["pseudo_label"]
    .values
)


pseudo_weights = (
    pseudo_labels_filtered_df["confidence"]
    .values
)



# ------------------------------------------------------------
# Concatenate modalities
# ------------------------------------------------------------

semi_tabular = np.concatenate(
    [
        human_tabular,
        pseudo_tabular
    ],
    axis=0
)


semi_text = np.concatenate(
    [
        human_text,
        pseudo_text
    ],
    axis=0
)


semi_graph = np.concatenate(
    [
        human_graph,
        pseudo_graph
    ],
    axis=0
)


semi_labels = np.concatenate(
    [
        human_labels,
        pseudo_labels
    ],
    axis=0
)


semi_weights = np.concatenate(
    [
        human_weights,
        pseudo_weights
    ],
    axis=0
)



# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

print("=" * 90)

print("Final Shapes:")


print(
    "Tabular:",
    semi_tabular.shape
)


print(
    "Text:",
    semi_text.shape
)


print(
    "Graph:",
    semi_graph.shape
)


print(
    "Labels:",
    semi_labels.shape
)


print(
    "Weights:",
    semi_weights.shape
)



print("\nLabel distribution:")

print(
    np.unique(
        semi_labels,
        return_counts=True
    )
)

BUILD FINAL SEMI-SUPERVISED ARRAYS
Final Shapes:
Tabular: (10171, 40)
Text: (10171, 768)
Graph: (10171, 3)
Labels: (10171,)
Weights: (10171,)

Label distribution:
(array([0, 1]), array([9911,  260]))


In [301]:
# ============================================================
# BUILD FINAL SEMI-SUPERVISED DATALOADER
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader



class SemiSupervisedFusionDataset(Dataset):

    def __init__(
        self,
        tabular,
        text,
        graph,
        labels,
        weights
    ):

        self.tabular = torch.tensor(
            tabular,
            dtype=torch.float32
        )

        self.text = torch.tensor(
            text,
            dtype=torch.float32
        )

        self.graph = torch.tensor(
            graph,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

        self.weights = torch.tensor(
            weights,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.labels)


    def __getitem__(self, idx):

        return {

            "tabular":
                self.tabular[idx],

            "text":
                self.text[idx],

            "graph":
                self.graph[idx],

            "label":
                self.labels[idx],

            "weight":
                self.weights[idx]
        }



semi_dataset = SemiSupervisedFusionDataset(

    tabular=semi_tabular,

    text=semi_text,

    graph=semi_graph,

    labels=semi_labels,

    weights=semi_weights
)



SEMI_BATCH_SIZE = 64


semi_loader = DataLoader(

    semi_dataset,

    batch_size=SEMI_BATCH_SIZE,

    shuffle=True
)



print("="*90)

print(
    "Dataset size:",
    len(semi_dataset)
)



batch = next(iter(semi_loader))


print("\nBatch shapes:")


print(
    "Tabular:",
    batch["tabular"].shape
)


print(
    "Text:",
    batch["text"].shape
)


print(
    "Graph:",
    batch["graph"].shape
)


print(
    "Labels:",
    batch["label"].shape
)


print(
    "Weights:",
    batch["weight"].shape
)

Dataset size: 10171

Batch shapes:
Tabular: torch.Size([64, 40])
Text: torch.Size([64, 768])
Graph: torch.Size([64, 3])
Labels: torch.Size([64])
Weights: torch.Size([64])


In [305]:
# ============================================================
# LOAD BASE MULTIMODAL MODEL
# ============================================================

import torch


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)



model = MultimodalFusionClassifier(
    tabular_dim=40,
    text_dim=768,
    graph_dim=3
)



checkpoint_path = (
    "..\\models\\multimodal_fusion_best.pt"
)



state_dict = torch.load(
    checkpoint_path,
    map_location=DEVICE
)



model.load_state_dict(
    state_dict
)



model = model.to(DEVICE)



print("="*90)

print(
    "Model loaded successfully"
)

print(
    "Device:",
    DEVICE
)

Model loaded successfully
Device: cpu


In [306]:
# ============================================================
# FORWARD PASS CHECK
# ============================================================

model.eval()


with torch.no_grad():

    batch = next(iter(semi_loader))


    tabular = batch["tabular"].to(DEVICE)

    text = batch["text"].to(DEVICE)

    graph = batch["graph"].to(DEVICE)



    logits = model(
        tabular,
        text,
        graph
    )



print("="*90)

print(
    "Forward pass successful"
)


print(
    "Input shapes:"
)

print(
    "Tabular:",
    tabular.shape
)

print(
    "Text:",
    text.shape
)

print(
    "Graph:",
    graph.shape
)


print(
    "\nOutput logits:"
)

print(
    logits.shape
)

Forward pass successful
Input shapes:
Tabular: torch.Size([64, 40])
Text: torch.Size([64, 768])
Graph: torch.Size([64, 3])

Output logits:
torch.Size([64, 2])


In [307]:
# ============================================================
# CLASS WEIGHTED LOSS + OPTIMIZER
# ============================================================

import torch
import torch.nn as nn


# Class weights
class_weights = torch.tensor(
    [
        1.0,   # Human
        5.0    # Bot
    ],
    dtype=torch.float32
).to(DEVICE)



criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    reduction="none"
)



optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-6,
    weight_decay=1e-4
)



print("="*90)

print(
    "Class weights:"
)

print(
    class_weights
)


print(
    "\nOptimizer:"
)

print(
    optimizer
)

Class weights:
tensor([1., 5.])

Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 1e-06
    maximize: False
    weight_decay: 0.0001
)


In [308]:
# ============================================================
# SEMI-SUPERVISED FINE-TUNING LOOP
# ============================================================

import torch
from tqdm import tqdm


EPOCHS = 20


best_loss = float("inf")


for epoch in range(EPOCHS):

    model.train()


    running_loss = 0.0

    correct = 0

    total = 0



    progress = tqdm(
        semi_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )



    for batch in progress:


        tabular = batch["tabular"].to(DEVICE)

        text = batch["text"].to(DEVICE)

        graph = batch["graph"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        weights = batch["weight"].to(DEVICE)



        optimizer.zero_grad()



        logits = model(
            tabular,
            text,
            graph
        )



        # per sample CE loss

        loss_values = criterion(
            logits,
            labels
        )



        # confidence weighting

        loss = (
            loss_values * weights
        ).mean()



        loss.backward()



        # stabilize training

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )



        optimizer.step()



        running_loss += (
            loss.item()
            *
            labels.size(0)
        )



        preds = torch.argmax(
            logits,
            dim=1
        )


        correct += (
            preds == labels
        ).sum().item()


        total += labels.size(0)



        progress.set_postfix(
            {
                "loss":
                f"{loss.item():.4f}"
            }
        )



    epoch_loss = (
        running_loss / total
    )


    epoch_acc = (
        correct / total
    )



    print("\n")
    print("="*90)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Loss: {epoch_loss:.6f}"
    )

    print(
        f"Accuracy: {epoch_acc:.4f}"
    )



    if epoch_loss < best_loss:


        best_loss = epoch_loss


        torch.save(
            model.state_dict(),
            "..\\models\\multimodal_fusion_semisupervised_best.pt"
        )


        print(
            "Best model saved"
        )

Epoch 1/20: 100%|██████████| 159/159 [00:04<00:00, 37.34it/s, loss=28315.7246]  




Epoch 1/20
Loss: 215624.261583
Accuracy: 0.9670
Best model saved


Epoch 2/20: 100%|██████████| 159/159 [00:03<00:00, 42.57it/s, loss=7718.2495]   




Epoch 2/20
Loss: 170726.035688
Accuracy: 0.9647
Best model saved


Epoch 3/20: 100%|██████████| 159/159 [00:02<00:00, 62.80it/s, loss=0.0008]      


...



Epoch 18/20
Loss: 221061.840098
Accuracy: 0.9657


Epoch 19/20: 100%|██████████| 159/159 [00:02<00:00, 61.36it/s, loss=42201.2617]  




Epoch 19/20
Loss: 200439.433233
Accuracy: 0.9656


Epoch 20/20: 100%|██████████| 159/159 [00:02<00:00, 67.95it/s, loss=4796.6802]   




Epoch 20/20
Loss: 164213.233175
Accuracy: 0.9659


In [309]:
# ============================================================
# CHECK LOGITS AFTER TRAINING
# ============================================================

model.eval()

with torch.no_grad():

    batch = next(iter(semi_loader))

    tabular = batch["tabular"].to(DEVICE)
    text = batch["text"].to(DEVICE)
    graph = batch["graph"].to(DEVICE)


    logits = model(
        tabular,
        text,
        graph
    )


print("="*90)

print(
    "Logits:"
)

print(
    logits[:10]
)


print(
    "\nMin:",
    logits.min().item()
)


print(
    "Max:",
    logits.max().item()
)

Logits:
tensor([[ -7.5448,   6.4173],
        [  6.5240,  -8.9035],
        [  6.2359,  -8.0115],
        [  8.9254, -11.4825],
        [  7.2792,  -9.4535],
        [ 10.4557, -13.3887],
        [ 10.4740, -12.9842],
        [ 12.7628, -15.6644],
        [ 10.7916, -13.6373],
        [ 11.9095, -14.6564]])

Min: -1533438.5
Max: 1417745.5


In [310]:
# ============================================================
# LOAD BEST SEMI-SUPERVISED MODEL
# ============================================================

best_path = (
    "..\\models\\multimodal_fusion_semisupervised_best.pt"
)


state_dict = torch.load(
    best_path,
    map_location=DEVICE
)


model.load_state_dict(
    state_dict
)


model.eval()


print("="*90)

print(
    "Best semi-supervised model loaded"
)

Best semi-supervised model loaded


In [311]:
print(val_tabular.shape)
print(val_text.shape)
print(val_graph.shape)
print(val_labels.shape)


print(test_tabular.shape)
print(test_text.shape)
print(test_graph.shape)
print(test_labels.shape)

(144, 40)
(144, 768)
(144, 3)
(144,)
(143, 40)
(143, 768)
(143, 3)
(143,)


In [312]:
# ============================================================
# EVALUATE SEMI-SUPERVISED MODEL
# ============================================================

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)



def evaluate_model(
    tabular,
    text,
    graph,
    labels,
    split_name
):

    model.eval()


    with torch.no_grad():

        tabular = torch.tensor(
            tabular,
            dtype=torch.float32
        ).to(DEVICE)


        text = torch.tensor(
            text,
            dtype=torch.float32
        ).to(DEVICE)


        graph = torch.tensor(
            graph,
            dtype=torch.float32
        ).to(DEVICE)



        logits = model(
            tabular,
            text,
            graph
        )


        probs = torch.softmax(
            logits,
            dim=1
        )[:,1].cpu().numpy()



    preds = (
        probs >= 0.5
    ).astype(int)



    print("="*90)

    print(
        split_name
    )

    print("="*90)


    print(
        "Accuracy:",
        accuracy_score(labels,preds)
    )

    print(
        "Precision:",
        precision_score(labels,preds,zero_division=0)
    )

    print(
        "Recall:",
        recall_score(labels,preds,zero_division=0)
    )

    print(
        "F1:",
        f1_score(labels,preds,zero_division=0)
    )

    print(
        "ROC-AUC:",
        roc_auc_score(labels,probs)
    )

    print(
        "PR-AUC:",
        average_precision_score(labels,probs)
    )


    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            labels,
            preds
        )
    )


    print("\nClassification Report:")

    print(
        classification_report(
            labels,
            preds,
            zero_division=0
        )
    )


    return probs, preds

In [313]:
val_probs, val_preds = evaluate_model(
    val_tabular,
    val_text,
    val_graph,
    val_labels,
    "VALIDATION - Semi Supervised"
)

VALIDATION - Semi Supervised
Accuracy: 0.6875
Precision: 0.22580645161290322
Recall: 0.25
F1: 0.23728813559322035
ROC-AUC: 0.521551724137931
PR-AUC: 0.20228494623655915

Confusion Matrix:
[[92 24]
 [21  7]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.79      0.80       116
           1       0.23      0.25      0.24        28

    accuracy                           0.69       144
   macro avg       0.52      0.52      0.52       144
weighted avg       0.70      0.69      0.69       144



In [314]:
test_probs, test_preds = evaluate_model(
    test_tabular,
    test_text,
    test_graph,
    test_labels,
    "TEST - Semi Supervised"
)

TEST - Semi Supervised
Accuracy: 0.7062937062937062
Precision: 0.29411764705882354
Recall: 0.35714285714285715
F1: 0.3225806451612903
ROC-AUC: 0.574223602484472
PR-AUC: 0.23091614268084854

Confusion Matrix:
[[91 24]
 [18 10]]

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.79      0.81       115
           1       0.29      0.36      0.32        28

    accuracy                           0.71       143
   macro avg       0.56      0.57      0.57       143
weighted avg       0.73      0.71      0.72       143



In [318]:
# ============================================================
# SAVE SEMI-SUPERVISED RESULTS FROM ACTUAL OUTPUT
# ============================================================

import pandas as pd
import os

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


results = []


for split_name, labels, probs, preds in [
    
    (
        "Semi-supervised Fusion Validation",
        val_labels,
        val_probs,
        val_preds
    ),
    
    (
        "Semi-supervised Fusion Test",
        test_labels,
        test_probs,
        test_preds
    )
]:

    results.append({

        "Model": split_name,

        "Accuracy":
            accuracy_score(
                labels,
                preds
            ),

        "Precision":
            precision_score(
                labels,
                preds,
                zero_division=0
            ),

        "Recall":
            recall_score(
                labels,
                preds,
                zero_division=0
            ),

        "F1":
            f1_score(
                labels,
                preds,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(
                labels,
                probs
            ),

        "PR_AUC":
            average_precision_score(
                labels,
                probs
            )
    })



semi_results_df = pd.DataFrame(
    results
)



os.makedirs(
    "../results",
    exist_ok=True
)


semi_results_df.to_csv(
    "../results/semi_supervised_results.csv",
    index=False
)



print("="*90)

print(
    "Semi-supervised results saved"
)


display(
    semi_results_df
)

Semi-supervised results saved


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Semi-supervised Fusion Validation,0.687500,0.225806,0.250000,0.237288,0.521552,0.202285
1,Semi-supervised Fusion Test,0.706294,0.294118,0.357143,0.322581,0.574224,0.230916


In [320]:
import torch
import os


os.makedirs(
    "../models",
    exist_ok=True
)


torch.save(
    model.state_dict(),
    "../models/multimodal_fusion_best.pt"
)


print("Base multimodal model saved")

Base multimodal model saved


In [321]:
import json


semi_metadata = {

    "experiment":
        "Semi-supervised V1",

    "base_model":
        "Multimodal Fusion",

    "pseudo_samples":
        9212,

    "human_samples":
        959,

    "modalities":
        [
            "tabular",
            "text",
            "graph"
        ],

    "training":
        "full_finetuning"

}


with open(
    "../final_data/semi_supervised/semi_supervised_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        semi_metadata,
        f,
        indent=4
    )


print(
    "Metadata saved"
)

Metadata saved


In [326]:
# ============================================================
# SAVE SEMI-SUPERVISED TEST PREDICTIONS
# ============================================================

import pandas as pd
import os


os.makedirs(
    "../results/predictions",
    exist_ok=True
)



semi_predictions_df = pd.DataFrame({

    "user_key": test_keys,

    "true_label": test_labels,

    "prediction": test_preds,

    "probability": test_probs

})



semi_predictions_df.to_csv(
    "../results/predictions/semi_supervised_test_predictions.csv",
    index=False
)



print("="*80)

print(
    "Saved Semi-supervised Test Predictions"
)

print(
    semi_predictions_df.shape
)


display(
    semi_predictions_df.head()
)

Saved Semi-supervised Test Predictions
(143, 4)


,user_key,true_label,prediction,probability
0,0dzisitpiinteke,0,1,1.0
1,abdorezaahmadi,0,0,0.0
2,achaemenid2582,0,0,0.0
3,alale26361947,1,0,0.0
4,alasht_iran,0,0,0.0
